In [12]:
import pandas as pd
from scipy.sparse import csr_matrix

# Load the Ratings CSV file
df_team49 = pd.read_csv('/Users/mahisidda/Downloads/IFT511/Project/Ratings.csv', delimiter=';')

# Create unique mappings for user IDs and book IDs
user_ids = df_team49['User-ID'].unique()
book_ids = df_team49['ISBN'].unique()

user_id_map = {user_id: idx for idx, user_id in enumerate(user_ids)}
book_id_map = {book_id: idx for idx, book_id in enumerate(book_ids)}

# Define the dimensions of the matrix
num_users = len(user_id_map)
num_books = len(book_id_map)

# Initialize lists for CSR matrix construction
row_indices = []
col_indices = []
ratings = []

# Populate the row indices, column indices, and ratings lists
for _, row in df_team49.iterrows():
    row_indices.append(user_id_map[row['User-ID']])
    col_indices.append(book_id_map[row['ISBN']])
    ratings.append(row['Rating'])

# Create the sparse user-book matrix
matrix_team49 = csr_matrix((ratings, (row_indices, col_indices)), shape=(num_users, num_books))

# Print the sparse and dense matrix formats for verification
print("Sparse User-Book Matrix:")
print(matrix_team49)

print("\nDense format (first 5 rows and columns):")
print(matrix_team49.toarray()[:5, :5])  # Print first 5 rows and columns

# Function to save the matrix in LibSVM format with y values as all zeros
def save_as_libsvm_with_y(sparse_matrix, filename):
    with open(filename, 'w') as f:
        for i in range(sparse_matrix.shape[0]):
            row = sparse_matrix[i, :]
            indices = row.nonzero()[1]
            values = row.data
            y_value = 0  # Placeholder y value (all zeros)
            line = f"{y_value} " + " ".join(f"{index}:{value}" for index, value in zip(indices, values))
            f.write(line + "\n")

# Save the matrix in LibSVM format with y values
save_as_libsvm_with_y(matrix_team49, '/Users/mahisidda/Downloads/IFT511/Project/49user_book_matrix_with_y.libsvm')

print("\nFile saved as '49user_book_matrix_with_y.libsvm'")


Sparse User-Book Matrix:
  (0, 0)	0
  (1, 1)	5
  (2, 2)	0
  (3, 3)	3
  (3, 4)	6
  (4, 5)	0
  (5, 6)	8
  (6, 7)	6
  (7, 8)	7
  (8, 9)	10
  (9, 10)	0
  (9, 11)	0
  (9, 12)	0
  (9, 13)	0
  (9, 14)	0
  (9, 15)	0
  (10, 16)	9
  (10, 17)	0
  (10, 18)	0
  (10, 19)	9
  (10, 20)	8
  (10, 21)	7
  (10, 22)	0
  (10, 23)	7
  (11, 24)	6
  :	:
  (105276, 117015)	0
  (105276, 255395)	0
  (105276, 340552)	0
  (105277, 146362)	0
  (105278, 2104)	0
  (105278, 3111)	6
  (105278, 8545)	0
  (105278, 8851)	7
  (105278, 12047)	0
  (105278, 20167)	0
  (105278, 26864)	0
  (105278, 34599)	0
  (105278, 46617)	0
  (105278, 50527)	0
  (105278, 50534)	5
  (105278, 56824)	0
  (105278, 195285)	0
  (105278, 226347)	9
  (105278, 284681)	0
  (105278, 340553)	0
  (105278, 340554)	5
  (105279, 7295)	0
  (105280, 12065)	10
  (105281, 78598)	10
  (105282, 340555)	8

Dense format (first 5 rows and columns):
[[0 0 0 0 0]
 [0 5 0 0 0]
 [0 0 0 0 0]
 [0 0 0 3 6]
 [0 0 0 0 0]]

File saved as '49user_book_matrix_with_y.libsvm'


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.datasets import load_svmlight_file

# Load the user-book matrix from the libsvm file
matrix, _ = load_svmlight_file('/Users/mahisidda/Downloads/IFT511/Project/49user_book_matrix_with_y.libsvm')

# Convert the sparse matrix to a DataFrame for easier manipulation
user_book_matrix = pd.DataFrame(matrix.toarray())

# Calculate cosine similarity between users
user_similarity = cosine_similarity(user_book_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_book_matrix.index, columns=user_book_matrix.index)

def recommend_books(user_id, k=10, n_recommendations=5):
    # Get the similarity scores for the user
    similar_users = user_similarity_df[user_id].nlargest(k + 1).index[1:]  # Exclude the user itself

    # Find books read by similar users
    books_read_by_similar_users = df_team49[df_team49['User-ID'].isin(similar_users)]['ISBN'].unique()

    # Calculate weighted average ratings for each book
    recommendations = {}
    for book in books_read_by_similar_users:
        ratings = df_team49[(df_team49['ISBN'] == book) & (df_team49['User-ID'].isin(similar_users))]
        if not ratings.empty:
            weighted_sum = (ratings['Rating'] * user_similarity_df.loc[ratings['User-ID'], user_id]).sum()
            similarity_sum = user_similarity_df.loc[ratings['User-ID'], user_id].sum()
            if similarity_sum > 0:
                recommendations[book] = weighted_sum / similarity_sum

    # Filter out books already read by the user
    books_read_by_user = df_team49[df_team49['User-ID'] == user_id]['ISBN'].unique()
    recommendations = {book: score for book, score in recommendations.items() if book not in books_read_by_user}

    # Get the top N recommendations
    recommended_books = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)[:n_recommendations]

    return recommended_books

# Generate recommendations for a specific user
user_id = 0  # Replace with the actual user index corresponding to the libsvm matrix
recommended_books = recommend_books(user_id)

# Prepare the results for CSV
results = []
for book_id, score in recommended_books:
    book_title = df_team49[df_team49['ISBN'] == book_id]['Book-Title'].values[0]  # Assuming 'Book-Title' is a column
    results.append({'User_ID': user_id, 'Book_ID': book_id, 'Book_Title': book_title, 'Recommendation_Score': score})

# Create a DataFrame and save to CSV
recommendation_df = pd.DataFrame(results)
recommendation_df.to_csv('49book_recommendations.csv', index=False)